In [2]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

print("Step 1: 라이브러리 로드 완료")

# 1. 사전 학습된 Faster R-CNN 모델 불러오기 (가장 성능이 좋은 기본 가중치 사용)
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)

# 2. 모델을 평가(추론)) 모드로 전환
model.eval()

print("모델 다운로드 및 로드 성공!")
print(f"이 모델이 알아볼 수 있는 객체 종류: {len(weights.meta['categories'])}가지")

Step 1: 라이브러리 로드 완료
모델 다운로드 및 로드 성공!
이 모델이 알아볼 수 있는 객체 종류: 91가지


In [3]:
import cv2
from PIL import Image

# 1. Faster R-CNN 전용 전처리 도구 가져오기
preprocess = weights.transforms()

# 2. 사진 준비
img_path = "../images/object_detection/my_photo.jpeg"
img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_pil = Image.fromarray(img_rgb)

# 3. 모델 입맛에 맞게 변환 (ToTensor + Normalization 등이 포함됨)
# unsqueeze(0)로 배치 차원을 추가(Faster R-CNN: 리스트 형태의 텐서 선호) 
input_batch = preprocess(img_pil).unsqueeze(0)

print(f"전처리 완료된 텐서 모양: {input_batch.shape}") # 예상 결과: [1, 3, H, W] 형태

전처리 완료된 텐서 모양: torch.Size([1, 3, 4000, 3000])


In [4]:
# 1. 모델과 데이터를 장치(MPS/CPU)로 보내기
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
input_batch = input_batch.to(device)

# 2. 추론 실행 (without 기울기 계산)
with torch.no_grad():
    prediction = model(input_batch)

# 3. 결과물(딕셔너리)의 열쇠(Keys) 확인
print(f"추론 결과 키 값: {prediction[0].keys()}")

추론 결과 키 값: dict_keys(['boxes', 'labels', 'scores'])


In [5]:
# 1. 텐서들을 CPU로 가져오기 (시각화나 분석을 위해)
boxes = prediction[0]['boxes'].cpu()
labels = prediction[0]['labels'].cpu()
scores = prediction[0]['scores'].cpu()

# 2. 상위 5개 결과 출력
print("--- Detection Top 5 Results ---")
for i in range(5):
    score = scores[i].item() # 텐서에서 숫자만 추출
    label_id = labels[i].item()
    category_name = weights.meta['categories'][label_id] # 번호를 이름으로 변환
    
    print(f"Top {i+1}: {category_name} (Confidence: {score:.2%})")

--- Detection Top 5 Results ---
Top 1: laptop (Confidence: 99.93%)
Top 2: mouse (Confidence: 99.83%)
Top 3: chair (Confidence: 98.94%)
Top 4: chair (Confidence: 98.50%)
Top 5: cup (Confidence: 96.11%)


In [9]:
import numpy as np

# 1. 시각화를 위한 원본 이미지 복사 
output_img = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR) # OpenCV는 BGR 사용

# 2. 필터링된 결과 그리기(신뢰도 0.8 이상)
# (1) 신뢰도 지정: 0.8
threshold = 0.8

# (2) 불리언 마스크(Boolean Mask) 생성: 조건에 맞는 인덱스만 True가 됨
keep_indices = scores > threshold

# (3) 필터링된 데이터 정의
filtered_boxes = boxes[keep_indices]
filtered_labels = labels[keep_indices]
filtered_scores = scores[keep_indices]

print(f"필터링 전: {len(boxes)}개 -> 필터링 후: {len(filtered_boxes)}개")

for box, label, score in zip(filtered_boxes, filtered_labels, filtered_scores):
    x1, y1, x2, y2 = box.int().numpy() # 좌표를 정수로 변환
    caption = f"{weights.meta['categories'][label]}: {score:.2%}"
    
    # 박스 그리기 (Color: Green, Thickness: 5)
    cv2.rectangle(output_img, (x1, y1), (x2, y2), (0, 255, 0), 5)
    
    # 라벨 텍스트 쓰기(Color: Green, Thickness: 3)
    cv2.putText(output_img, caption, (x1, y1 - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

# 3. 결과 저장 및 확인
cv2.imwrite("../images/object_detection/result_macbook.jpg", output_img)
print("결과 이미지가 'result_macbook.jpg'로 저장되었습니다!")

필터링 전: 52개 -> 필터링 후: 8개
결과 이미지가 'result_macbook.jpg'로 저장되었습니다!
